# Colony counting pipeline

Isolate each well from a multi-well plate `.tif` scan, then run each well crop through Cellpose to count colonies and estimate diameter. Source scans live in `../Clonogenics`.

In [ ]:
import os
import glob
import shutil
import warnings
from datetime import datetime
warnings.filterwarnings("ignore", message="Sparse invariant checks")

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile as tifi
import torch
from scipy import ndimage
from tqdm.notebook import tqdm
from ultralytics import FastSAM
from cellpose import models

# Make our inline plots look nice and big
plt.rcParams['figure.figsize'] = [12, 6]

## 1. Config — shared tuning

Plate geometry (which plates, how many wells, where) is **not** configured here anymore — it comes from ROI hints you draw in Section 3c. This cell only holds shared image-processing constants (CLAHE/Canny for the contour detector, the Hough refinement block, Cellpose segmentation params) plus run settings (`BATCH_MODE` / `WELLS_ONLY` / `input_path` / `BATCH_OUTPUT_DIR`).

In [ ]:
# ============================================================
# Run settings — what this notebook run actually does.
# ============================================================
BATCH_MODE = True     # False: run a single file (input_path). True: sweep the ROI series.
WELLS_ONLY = True    # True: stop after well-detection, skip the slower Cellpose step.
input_path = os.path.join("../Clonogenics-orig", "4h HS 43 + radiation001.tif")  # BATCH_MODE=False
BATCH_OUTPUT_DIR = "batch_output"  # used when BATCH_MODE=True — one subfolder per plate

# ============================================================
# Shared image-processing constants (scan contrast/edge tuning). CLAHE/Canny feed the
# scoped contour detector; the REFINE_*/HOUGH_* block feeds refine_well (per-well ring
# snap). Only re-tune if detection misbehaves on a new scan style.
# ============================================================
CLAHE_CLIP = 3.0
CANNY_LOW, CANNY_HIGH = 30, 90

# Per-well edge refinement (local Hough): the analytic layout is close but not
# pixel-perfect; snap each circle to the real ring via Hough in a small ROI around the
# computed center, constrained near the known radius. Falls back to the analytic circle
# if no good ring is found. Downscaling the ROI before searching (then scaling the found
# circle back up) gives ~12x speedup for a few px of localization noise.
REFINE_WELLS        = True
REFINE_SEARCH_FRAC  = 0.5    # ROI padding beyond the well radius, as a fraction of radius
REFINE_RADIUS_TOL   = 0.2    # Hough searches radii within +/- this fraction of the analytic r
REFINE_MAX_SHIFT    = 0.5    # reject a refined center that moved > this fraction of r (bad lock)
REFINE_DOWNSCALE_PX = 300    # shrink the ROI so its longer side is about this many pixels
HOUGH_PARAM1 = 100
HOUGH_PARAM2 = 30

# ============================================================
# Cellpose colony segmentation. Tune these if colonies look under/over-segmented (two
# touching colonies merged into one blob, or one colony split into two).
# ============================================================
# Expected colony diameter in px, in the LAB-distance "signal" image fed to Cellpose (not
# the raw well crop). Cellpose uses this to scale its internal model — too large merges
# nearby colonies, too small can shatter one colony into several.
CELLPOSE_DIAMETER = 70
# Max allowed flow-reconstruction error per mask (Cellpose's internal QC score). Higher =
# keep more masks, including rougher/noisier ones; lower = reject malformed masks more
# aggressively (fewer false positives, but can also drop real irregular colonies).
CELLPOSE_FLOW_THRESHOLD = 0.9
# A pixel is called "part of a colony" where the model's cell-probability map exceeds this.
# Lower (more negative) = more permissive, catches faint/sparse colonies but risks noise;
# higher = stricter, cleaner background but can miss faint real colonies.
CELLPOSE_CELLPROB_THRESHOLD = -2.0
# Percentile range used to normalize the signal image's intensity before segmentation —
# clips extreme outlier pixels so one bright artifact doesn't wash out the contrast Cellpose
# needs to see real colonies.
CELLPOSE_NORMALIZE_PERCENTILE = [1.0, 99.0]
# Non-colony brightness outlier rejection, in LAB L (lightness) units, relative to the local
# well-background L — not tied to position, since glints/debris can land anywhere in a well.
# Crystal-violet colonies sit in a mid-darkness band: dirt/hair/specks are much darker
# (near-black) than any real colony, while a specular reflection/glint is brighter than
# background. (dark_margin, bright_margin): a pixel darker than background by more than
# dark_margin, or brighter than background by more than bright_margin, is zeroed out of the
# Cellpose signal image before segmentation. Raise dark_margin if real dense/dark colonies
# start getting stripped; raise bright_margin if real bright-background wells start losing
# edge pixels; lower either if debris/glint still leaks through as false colonies.
L_OUTLIER_MARGIN = (90, 40)  # (dark_margin, bright_margin)

print("Config loaded. Plate geometry now comes from ROI hints (Section 3c), not a profile.")

## 2. Pick device & load models

Picks `mps`/`cuda`/`cpu` automatically (portable between Apple Silicon and the 1650 Ti box), loads FastSAM and Cellpose (colony counter) once. Reused by both single-image and batch runs below.

In [ ]:
# Portable device pick: mps on Apple Silicon, cuda on the 1650 Ti box, cpu fallback.
# DEVICE (str) feeds ultralytics/FastSAM; TORCH_DEVICE (torch.device) feeds Cellpose.
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
TORCH_DEVICE = torch.device(DEVICE)
print(f"Using device: {DEVICE}")

print("Loading FastSAM (Well Extractor) into GPU...")
sam_model = FastSAM('FastSAM-s.pt')

print("Loading Cellpose (Colony Counter) into GPU...")
cp_model = models.CellposeModel(gpu=DEVICE != "cpu", device=TORCH_DEVICE, model_type='cpsam_v2')
# Directly confirm what device the model actually landed on — don't infer this from
# whether Cellpose's own log messages showed up, since those need io.logger_setup()
# (which also drives a per-tile progress bar) to be visible at all.
print(f"cp_model.device = {cp_model.device}, cp_model.gpu = {cp_model.gpu}")
print("Done!")

## 3. Detect wells (ROI-hinted)

Well detection is driven by the ROI hints drawn in Section 3c — no per-dataset profile, no global crop. `refine_well` below (the per-well Hough ring-snap) is the one piece kept from the old approach; it is reused by the ROI pipeline in Section 3b/3c. Run this cell to define it.

In [ ]:
def refine_well(gray, cx, cy, r):
    """Snap an analytic (cx, cy, r) to the real well ring with a local Hough search.

    gray is the full-image grayscale. Runs Hough on a downscaled ROI (much faster on these
    high-res scans) and scales the result back up. Returns the refined (cx, cy, r), or the
    input unchanged if no ring near the prior is found. Reused by the ROI-hinted path below.
    """
    pad = int(r * (1 + REFINE_SEARCH_FRAC))
    height, width = gray.shape
    x0, x1 = max(0, cx - pad), min(width, cx + pad)
    y0, y1 = max(0, cy - pad), min(height, cy + pad)
    roi = gray[y0:y1, x0:x1]
    if roi.size == 0:
        return cx, cy, r

    roi = cv2.medianBlur(roi, 5)
    scale = min(1.0, REFINE_DOWNSCALE_PX / max(roi.shape))
    search_roi = cv2.resize(roi, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA) if scale < 1.0 else roi

    circles = cv2.HoughCircles(
        search_roi, cv2.HOUGH_GRADIENT, dp=1.2,
        minDist=max(search_roi.shape),          # only expect one ring in this ROI
        param1=HOUGH_PARAM1, param2=HOUGH_PARAM2,
        minRadius=int(r * (1 - REFINE_RADIUS_TOL) * scale),
        maxRadius=int(r * (1 + REFINE_RADIUS_TOL) * scale),
    )
    if circles is None:
        return cx, cy, r
    if scale < 1.0:
        circles = circles / scale  # back to ROI (full-res) coordinates

    # pick the ring whose center is closest to the analytic prior (ROI center)
    roi_cx, roi_cy = cx - x0, cy - y0
    found_cx, found_cy, found_r = min(
        circles[0], key=lambda cir: (cir[0] - roi_cx) ** 2 + (cir[1] - roi_cy) ** 2
    )
    if (found_cx - roi_cx) ** 2 + (found_cy - roi_cy) ** 2 > (r * REFINE_MAX_SHIFT) ** 2:
        return cx, cy, r  # moved too far — likely locked onto a colony/edge, keep analytic

    return int(x0 + found_cx), int(y0 + found_cy), int(found_r)

## 3b. ROI-hinted plate detection

The user draws one box per plate **on the true plate outline** (`jupyter-bbox-widget`, Section 3c); code pads it by `MARGIN_FRAC` and re-detects the precise plate box per image, so hand-placement shift between scans is absorbed. Wells are placed by an even `rows×cols` grid inside the detected box, then snapped to rings by `refine_well` (Section 3).

Two detectors are raced (pick the winner in the bake-off cell, then set `DETECTION_METHOD` in Section 5):
- **A — contour** (`detect_plate_rect_contour`): CLAHE→Canny→close→contour, scoped to the padded ROI.
- **B — edge-snap** (`detect_plate_rect_edges`): sums gradient strength *along* each edge direction and snaps each of the 4 sides to its strongest line; any side with no clear peak keeps the drawn edge. More robust to faint borders + interior clutter.

In [ ]:
def well_grid_fracs(rows, cols):
    """Even-spaced cell-center fractions inside a plate box (center of each grid cell)."""
    x_fracs = [(col_index + 0.5) / cols for col_index in range(cols)]
    y_fracs = [(row_index + 0.5) / rows for row_index in range(rows)]
    return x_fracs, y_fracs


def roi_frac_to_px(roi, image_width, image_height):
    """Convert a fractional {x,y,w,h} box (each in [0,1]) to integer (x0,y0,x1,y1) pixels."""
    x0 = int(round(roi["x"] * image_width))
    y0 = int(round(roi["y"] * image_height))
    x1 = int(round((roi["x"] + roi["w"]) * image_width))
    y1 = int(round((roi["y"] + roi["h"]) * image_height))
    return x0, y0, x1, y1


def pad_box(box_px, image_width, image_height, margin_frac):
    """Expand a pixel box by margin_frac of its own width/height on each side, clamped."""
    x0, y0, x1, y1 = box_px
    pad_x = int(round((x1 - x0) * margin_frac))
    pad_y = int(round((y1 - y0) * margin_frac))
    return (
        max(0, x0 - pad_x),
        max(0, y0 - pad_y),
        min(image_width, x1 + pad_x),
        min(image_height, y1 + pad_y),
    )

In [ ]:
def snap_edge(strength_profile, prior_index, radius, min_peak_ratio):
    """Snap to the strong edge NEAREST the drawn (prior) edge within +/- radius.

    Since the user draws accurately on the true plate, the plate's own wall is the closest
    strong edge to the drawn line; a neighbouring plate's wall (or the handwriting plate to
    the side) is farther, so 'nearest strong edge' rejects it even when it is brighter.
    A column/row counts as an edge if its summed gradient exceeds min_peak_ratio x the
    profile median. Falls back to the drawn edge if nothing in the window qualifies.
    """
    profile_median = float(np.median(strength_profile)) or 1.0
    threshold = min_peak_ratio * profile_median
    low = max(0, prior_index - radius)
    high = min(len(strength_profile), prior_index + radius + 1)
    strong_indices = [index for index in range(low, high) if strength_profile[index] > threshold]
    if not strong_indices:
        return prior_index
    return min(strong_indices, key=lambda index: abs(index - prior_index))


def detect_plate_rect_edges(gray, search_box, prior_box, min_peak_ratio=2.0):
    """Approach B: snap each of the four plate edges to the strong line NEAREST the drawn edge.

    Sums |gradient| along each edge direction (down columns for the vertical left/right edges,
    across rows for the horizontal top/bottom edges). Each edge is searched only within +/- the
    padding on that side (the shift tolerance) and snapped to the nearest strong line, so it
    can't jump to a neighbouring plate or the handwriting plate. Returns (x, y, w, h) in
    full-image pixels; any side with no strong edge keeps the drawn edge.
    """
    search_x0, search_y0, search_x1, search_y1 = search_box
    prior_x0, prior_y0, prior_x1, prior_y1 = prior_box
    crop = gray[search_y0:search_y1, search_x0:search_x1]

    gradient_x = np.abs(cv2.Sobel(crop, cv2.CV_64F, 1, 0, ksize=3))
    gradient_y = np.abs(cv2.Sobel(crop, cv2.CV_64F, 0, 1, ksize=3))
    column_strength = gradient_x.sum(axis=0)   # one value per column -> vertical edges
    row_strength = gradient_y.sum(axis=1)      # one value per row    -> horizontal edges

    # search radius on each side = the padding on that side (i.e. the shift tolerance)
    radius_left = max(1, prior_x0 - search_x0)
    radius_right = max(1, search_x1 - prior_x1)
    radius_top = max(1, prior_y0 - search_y0)
    radius_bottom = max(1, search_y1 - prior_y1)

    left_local = snap_edge(column_strength, prior_x0 - search_x0, radius_left, min_peak_ratio)
    right_local = snap_edge(column_strength, prior_x1 - search_x0, radius_right, min_peak_ratio)
    top_local = snap_edge(row_strength, prior_y0 - search_y0, radius_top, min_peak_ratio)
    bottom_local = snap_edge(row_strength, prior_y1 - search_y0, radius_bottom, min_peak_ratio)

    left_x = search_x0 + left_local
    right_x = search_x0 + right_local
    top_y = search_y0 + top_local
    bottom_y = search_y0 + bottom_local
    return left_x, top_y, right_x - left_x, bottom_y - top_y

In [ ]:
def detect_plate_rect_contour(gray, search_box, clahe_clip=CLAHE_CLIP, canny_low=CANNY_LOW,
                              canny_high=CANNY_HIGH, rectangularity_min=0.6,
                              min_area_frac=0.05):
    """Approach A: reuse the CLAHE -> Canny -> close -> contour machinery, scoped to the ROI.

    Expects exactly one plate in search_box: returns the largest boxy contour's bounding box
    (in full-image pixels) whose fill ratio >= rectangularity_min and whose area is at least
    min_area_frac of the crop. Returns None if nothing qualifies.
    """
    search_x0, search_y0, search_x1, search_y1 = search_box
    crop = gray[search_y0:search_y1, search_x0:search_x1]
    crop_area = crop.shape[0] * crop.shape[1]

    contrast = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=(8, 8)).apply(crop)
    blurred = cv2.GaussianBlur(contrast, (7, 7), 0)
    edges = cv2.Canny(blurred, canny_low, canny_high)
    edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, np.ones((25, 25), np.uint8))

    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    best_box = None
    best_area = 0.0
    for contour in contours:
        contour_area = cv2.contourArea(contour)
        rect_x, rect_y, rect_width, rect_height = cv2.boundingRect(contour)
        bounding_area = rect_width * rect_height
        if bounding_area == 0:
            continue
        rectangularity = contour_area / bounding_area
        if rectangularity < rectangularity_min:
            continue
        if bounding_area < min_area_frac * crop_area:
            continue
        if bounding_area > best_area:
            best_area = bounding_area
            best_box = (search_x0 + rect_x, search_y0 + rect_y, rect_width, rect_height)
    return best_box

In [ ]:
def detect_plate_rect(gray, search_box, prior_box, method="edges", **params):
    """Dispatch to the chosen detector; always return an (x, y, w, h) box."""
    if method == "edges":
        return detect_plate_rect_edges(gray, search_box, prior_box, **params)
    if method == "contour":
        detected = detect_plate_rect_contour(gray, search_box, **params)
        if detected is not None:
            return detected
        prior_x0, prior_y0, prior_x1, prior_y1 = prior_box
        return prior_x0, prior_y0, prior_x1 - prior_x0, prior_y1 - prior_y0
    raise ValueError(f"unknown detection method: {method!r}")


def reconcile_vertical_borders(gray, plate_boxes, prior_boxes, margin_frac=0.05, min_peak_ratio=2.0):
    """Make vertically-stacked plates share one hard border, so they never gap or overlap.

    The two plates sit in one molded tray with a single rib between them. Detecting each
    plate's inner edge independently lets them disagree on a shifted scan (gap or overlap).
    Instead, for each adjacent pair we find the single strongest horizontal edge (the rib)
    in the band between their drawn inner edges (widened by the shift tolerance) and set the
    upper plate's bottom = the lower plate's top = that line. Mirrors the old detector's
    awareness of the hard border between stacked plates. Returns updated (x, y, w, h) boxes.
    """
    order = sorted(range(len(plate_boxes)), key=lambda index: plate_boxes[index][1])
    boxes = [list(box) for box in plate_boxes]
    for upper_index, lower_index in zip(order, order[1:]):
        # shared horizontal extent of the two boxes (only look at columns they both cover)
        shared_x0 = max(boxes[upper_index][0], boxes[lower_index][0])
        shared_x1 = min(boxes[upper_index][0] + boxes[upper_index][2],
                        boxes[lower_index][0] + boxes[lower_index][2])
        if shared_x1 <= shared_x0:
            continue

        drawn_upper_bottom = prior_boxes[upper_index][3]
        drawn_lower_top = prior_boxes[lower_index][1]
        upper_drawn_height = prior_boxes[upper_index][3] - prior_boxes[upper_index][1]
        lower_drawn_height = prior_boxes[lower_index][3] - prior_boxes[lower_index][1]
        radius = int(max(upper_drawn_height, lower_drawn_height) * margin_frac)
        band_lo = max(0, min(drawn_upper_bottom, drawn_lower_top) - radius)
        band_hi = min(gray.shape[0], max(drawn_upper_bottom, drawn_lower_top) + radius)
        default_border_y = (drawn_upper_bottom + drawn_lower_top) // 2

        band = gray[band_lo:band_hi, shared_x0:shared_x1]
        if band.size == 0:
            border_y = default_border_y
        else:
            gradient_y = np.abs(cv2.Sobel(band, cv2.CV_64F, 0, 1, ksize=3))
            row_strength = gradient_y.sum(axis=1)
            profile_median = float(np.median(row_strength)) or 1.0
            peak = int(np.argmax(row_strength))
            border_y = band_lo + peak if row_strength[peak] > min_peak_ratio * profile_median else default_border_y

        # snap the shared edge: upper plate ends at border_y, lower plate starts at border_y
        boxes[upper_index][3] = max(1, border_y - boxes[upper_index][1])
        lower_bottom = boxes[lower_index][1] + boxes[lower_index][3]
        boxes[lower_index][1] = border_y
        boxes[lower_index][3] = max(1, lower_bottom - border_y)
    return [tuple(box) for box in boxes]


def place_wells(gray, refine_gray, plate_box, rows, cols, well_r_frac, plate_letter, refine=True):
    """Even rows x cols grid of wells inside plate_box, each snapped by the existing refine_well."""
    plate_x, plate_y, plate_width, plate_height = plate_box
    x_fracs, y_fracs = well_grid_fracs(rows, cols)
    base_radius = int(plate_width * well_r_frac)

    wells = []
    for row_index, y_frac in enumerate(y_fracs):
        for col_index, x_frac in enumerate(x_fracs):
            center_x = int(plate_x + plate_width * x_frac)
            center_y = int(plate_y + plate_height * y_frac)
            if refine:
                center_x, center_y, radius = refine_well(refine_gray, center_x, center_y, base_radius)
            else:
                radius = base_radius
            label = f"{plate_letter}{row_index * cols + col_index + 1}"
            wells.append((center_x, center_y, radius, label))
    return wells


def detect_wells_from_rois(image_rgb, roi_hints, method="edges", margin_frac=0.05,
                           well_r_frac=0.20, refine=True, return_boxes=False):
    """ROI-hinted replacement for detect_plate_rects + x_limit_frac: detect a plate box per
    drawn ROI, reconcile the shared border between stacked plates, place wells, snap to rings.
    Returns (x, y, r, label) for every well.

    return_boxes=True also returns the detected (x, y, w, h) plate box per ROI, so overlays
    can show what the detector actually locked onto (vs the padded search area).
    """
    image_height, image_width = image_rgb.shape[:2]
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY) if image_rgb.ndim == 3 else image_rgb
    refine_gray = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=(8, 8)).apply(gray) if refine else gray

    # Pass 1: detect each plate box independently within its padded ROI.
    prior_boxes = [roi_frac_to_px(roi, image_width, image_height) for roi in roi_hints]
    plate_boxes = []
    for prior_box in prior_boxes:
        search_box = pad_box(prior_box, image_width, image_height, margin_frac)
        plate_boxes.append(detect_plate_rect(gray, search_box, prior_box, method=method))

    # Pass 2: make vertically-stacked plates share the hard border (no gap/overlap).
    if len(plate_boxes) > 1:
        plate_boxes = reconcile_vertical_borders(gray, plate_boxes, prior_boxes, margin_frac)

    # Pass 3: place + refine wells inside the reconciled boxes.
    all_wells = []
    for roi_index, (roi, plate_box) in enumerate(zip(roi_hints, plate_boxes)):
        plate_letter = roi.get("letter") or chr(ord("A") + roi_index)
        all_wells.extend(
            place_wells(gray, refine_gray, plate_box, roi["rows"], roi["cols"],
                        well_r_frac, plate_letter, refine=refine)
        )
    if return_boxes:
        return all_wells, plate_boxes
    return all_wells

## 3c. ROI bake-off — draw ROIs, race A vs B against gold

**How to run:**
1. Run the config cell, then the bbox cell — **draw one box per plate** on the true plate outline (top plate first, then bottom), then run the `roi_hints` cell.
2. Run the bake-off cell: each shifted scan renders **edge-snap (B) beside contour (A)**; printed well counts should equal `expected_well_count` (12) for whichever method placed all plates.
3. Run the gold cell to compare against the prior pipeline's `grid.png`.
4. Eyeball across all 5 shifted scans — which method keeps wells centered despite hand-placement shift? Record the winner in the markdown cell at the end.

In [ ]:
# Validation dataset for the ROI-hinted detection prototype.
ROI_INPUT_DIR = "../Clonogenics-orig"
SHIFT_SERIES = [
    "4h HS 43 + radiation001.tif",
    "4h HS 43 + radiation002.tif",
    "4h HS 43 + radiation004.tif",
    "4h HS 43 + radiation005.tif",
    "4h HS 43 + radiation006.tif",
]
GOLD_DIR = "batch_output_20260704_163617"   # gold grid.png per plate for comparison

MARGIN_FRAC = 0.05              # ROI padding to absorb hand-placement shift
PLATE_ROWS, PLATE_COLS = 3, 2   # this dataset: two 3x2 plates
WELL_R_FRAC = 0.20

In [ ]:
from jupyter_bbox_widget import BBoxWidget

# Load the first scan, downscale to a displayable 8-bit preview for the widget.
first_scan = tifi.imread(os.path.join(ROI_INPUT_DIR, SHIFT_SERIES[0]))
first_rgb = cv2.cvtColor(first_scan, cv2.COLOR_GRAY2RGB) if first_scan.ndim == 2 else first_scan[..., :3]
if first_rgb.dtype != np.uint8:
    first_rgb = cv2.normalize(first_rgb, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

preview_scale = 1200 / max(first_rgb.shape[0], first_rgb.shape[1])
preview = cv2.resize(first_rgb, None, fx=preview_scale, fy=preview_scale, interpolation=cv2.INTER_AREA)
preview_path = "roi_preview.png"
cv2.imwrite(preview_path, cv2.cvtColor(preview, cv2.COLOR_RGB2BGR))

# hide_buttons=True drops the Submit/Skip buttons (we read bbox_widget.bboxes directly,
# no submit callback needed).
bbox_widget = BBoxWidget(image=preview_path, classes=["plate"], hide_buttons=True)
bbox_widget

In [ ]:
# Convert preview-pixel boxes to image fractions, top-to-bottom, labeled A, B, ...
drawn_boxes = sorted(bbox_widget.bboxes, key=lambda box: box["y"])
roi_hints = []
for plate_index, box in enumerate(drawn_boxes):
    roi_hints.append({
        "x": box["x"] / preview.shape[1],
        "y": box["y"] / preview.shape[0],
        "w": box["width"] / preview.shape[1],
        "h": box["height"] / preview.shape[0],
        "rows": PLATE_ROWS,
        "cols": PLATE_COLS,
        "letter": chr(ord("A") + plate_index),
    })
print(f"Captured {len(roi_hints)} ROI hints:")
for roi in roi_hints:
    print(roi)

In [ ]:
def render_overlay(axis, image_rgb, roi_hints, wells, title, plate_boxes=None):
    """Overlay on `axis`: padded search area (yellow), detected plate rect (lime,
    if plate_boxes given), well circles (red) + labels."""
    axis.imshow(image_rgb)
    axis.set_title(title)
    image_height, image_width = image_rgb.shape[:2]
    for roi in roi_hints:
        prior_box = roi_frac_to_px(roi, image_width, image_height)
        search_x0, search_y0, search_x1, search_y1 = pad_box(prior_box, image_width, image_height, MARGIN_FRAC)
        axis.add_patch(plt.Rectangle((search_x0, search_y0), search_x1 - search_x0,
                                     search_y1 - search_y0, fill=False, edgecolor="yellow", linewidth=1.5))
    if plate_boxes:
        for plate_x, plate_y, plate_width, plate_height in plate_boxes:
            axis.add_patch(plt.Rectangle((plate_x, plate_y), plate_width, plate_height,
                                         fill=False, edgecolor="lime", linewidth=2))
    for center_x, center_y, radius, label in wells:
        axis.add_patch(plt.Circle((center_x, center_y), radius, fill=False, edgecolor="red", linewidth=2))
        axis.text(center_x + 10, center_y + 10, label, color="yellow", fontsize=12, fontweight="bold")
    axis.set_aspect("equal")

In [ ]:
expected_well_count = sum(roi["rows"] * roi["cols"] for roi in roi_hints)

for scan_name in SHIFT_SERIES:
    scan = tifi.imread(os.path.join(ROI_INPUT_DIR, scan_name))
    scan_rgb = cv2.cvtColor(scan, cv2.COLOR_GRAY2RGB) if scan.ndim == 2 else scan[..., :3]

    wells_edges, boxes_edges = detect_wells_from_rois(scan_rgb, roi_hints, method="edges",
                                                      margin_frac=MARGIN_FRAC, well_r_frac=WELL_R_FRAC,
                                                      return_boxes=True)
    wells_contour, boxes_contour = detect_wells_from_rois(scan_rgb, roi_hints, method="contour",
                                                          margin_frac=MARGIN_FRAC, well_r_frac=WELL_R_FRAC,
                                                          return_boxes=True)

    figure, (axis_edges, axis_contour) = plt.subplots(1, 2, figsize=(20, 12))
    render_overlay(axis_edges, scan_rgb, roi_hints, wells_edges,
                   f"{scan_name}  -  B: edge-snap", plate_boxes=boxes_edges)
    render_overlay(axis_contour, scan_rgb, roi_hints, wells_contour,
                   f"{scan_name}  -  A: contour", plate_boxes=boxes_contour)
    plt.show()

    print(f"{scan_name}: edges wells={len(wells_edges)}, contour wells={len(wells_contour)}, "
          f"expected={expected_well_count}   [yellow=search area, lime=detected plate, red=wells]")

In [ ]:
from PIL import Image

for scan_name in SHIFT_SERIES:
    gold_grid = os.path.join(GOLD_DIR, scan_name, "grid.png")
    if os.path.exists(gold_grid):
        figure, axis = plt.subplots(figsize=(10, 12))
        axis.imshow(Image.open(gold_grid))
        axis.set_title(f"GOLD: {scan_name}")
        axis.axis("off")
        plt.show()
    else:
        print(f"no gold grid for {scan_name}")

## 4. Segment colonies per well (Cellpose)

Defines `count_colonies(img_rgb, valid_wells, plate_name, show_plots=True)`: for each detected well, crops to a circle, builds a color-agnostic "distance from background" signal in LAB space, then runs Cellpose to segment/count colonies. Returns a list of `{"Plate", "Well", "Colonies"}` rows. Works off whatever wells the ROI-hinted detection found.

In [ ]:
def count_colonies(img_rgb, valid_wells, plate_name, show_plots=True, save_dir=None):
    """Segment colonies in each detected well and return per-well colony counts.

    show_plots displays each well's raw/segmentation panel inline; save_dir (if given)
    writes one PNG per well to disk instead/as well.
    """
    report_data = []
    print(f"Extracting wells and generating colony masks for {plate_name}...\n")

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)

    dark_margin, bright_margin = L_OUTLIER_MARGIN

    well_progress = tqdm(valid_wells, desc=plate_name, unit="well")
    for (x, y, r, label) in well_progress:
        well_progress.set_postfix(well=label)
        # Crop from img_rgb (known-good color), NOT the raw img
        crop_r = int(r * 0.94)
        y_min, y_max = max(0, y - crop_r), min(img_rgb.shape[0], y + crop_r)
        x_min, x_max = max(0, x - crop_r), min(img_rgb.shape[1], x + crop_r)

        well_crop = img_rgb[y_min:y_max, x_min:x_max].copy()   # RGB

        # Circular mask
        mask = np.zeros(well_crop.shape[:2], dtype="uint8")
        cv2.circle(mask, (well_crop.shape[1] // 2, well_crop.shape[0] // 2), crop_r, 255, -1)
        final_well = cv2.bitwise_and(well_crop, well_crop, mask=mask)   # RGB

        # --- Color-agnostic signal: distance from background in LAB ---
        lab = cv2.cvtColor(final_well, cv2.COLOR_RGB2LAB).astype(np.float32)

        ring = cv2.subtract(mask, cv2.erode(mask, np.ones((60, 60), np.uint8)))
        bg_a = np.median(lab[:, :, 1][ring > 0])
        bg_b = np.median(lab[:, :, 2][ring > 0])
        bg_L = np.median(lab[:, :, 0][ring > 0])

        dist = np.sqrt((lab[:, :, 1] - bg_a) ** 2 + (lab[:, :, 2] - bg_b) ** 2)
        signal = cv2.normalize(dist, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        signal = cv2.bitwise_and(signal, signal, mask=mask)

        # Non-colony brightness outlier rejection (debris too dark, glint too bright) —
        # see L_OUTLIER_MARGIN in Config for the reasoning. Zeroed out of the Cellpose
        # signal image before segmentation so neither gets counted as a colony.
        dark_mask = (lab[:, :, 0] < (bg_L - dark_margin)) & (mask > 0)
        bright_mask = (lab[:, :, 0] > (bg_L + bright_margin)) & (mask > 0)
        signal[dark_mask | bright_mask] = 0

        # Visual check for L_OUTLIER_MARGIN tuning: tint whichever pixels got rejected
        # directly on the raw well crop — red for debris (too dark), cyan for glint (too
        # bright) — so it's obvious before Cellpose ever sees them, no LAB math required
        # to read it.
        outlier_overlay = final_well.copy()
        outlier_overlay[dark_mask] = [255, 0, 0]
        outlier_overlay[bright_mask] = [0, 255, 255]

        # --- Cellpose ---
        masks_cp, flows, styles = cp_model.eval(
            signal,
            diameter=CELLPOSE_DIAMETER,
            flow_threshold=CELLPOSE_FLOW_THRESHOLD,
            cellprob_threshold=CELLPOSE_CELLPROB_THRESHOLD,
            normalize={"normalize": True, "percentile": CELLPOSE_NORMALIZE_PERCENTILE},
        )

        colony_count = int(masks_cp.max())
        report_data.append({
            "Plate": plate_name,
            "Well": label,
            "Colonies": colony_count
        })

        if show_plots or save_dir:
            fig, axes = plt.subplots(1, 3)
            axes[0].imshow(final_well)          # already RGB — no conversion
            axes[0].set_title(f"Raw Well: {label}")
            axes[0].axis('off')

            axes[1].imshow(outlier_overlay)
            axes[1].set_title("Outliers (red=debris, cyan=glint)")
            axes[1].axis('off')

            axes[2].imshow(signal, cmap='magma')
            axes[2].imshow(masks_cp, cmap='nipy_spectral', alpha=0.4)
            axes[2].set_title(f"AI Count: {colony_count} Colonies")
            axes[2].axis('off')

            # number each colony at its centroid so a vibe-check is fast — did Cellpose
            # merge two touching colonies into one, or split one into two?
            colony_ids = np.unique(masks_cp)
            colony_ids = colony_ids[colony_ids != 0]
            if len(colony_ids) > 0:
                centroids = ndimage.center_of_mass(masks_cp, masks_cp, colony_ids)
                for colony_id, (centroid_y, centroid_x) in zip(colony_ids, centroids):
                    axes[2].text(centroid_x, centroid_y, str(int(colony_id)),
                                color='white', fontsize=6, ha='center', va='center',
                                bbox=dict(boxstyle='round,pad=0.1', facecolor='black', alpha=0.5, linewidth=0))

            if save_dir:
                well_path = os.path.join(save_dir, f"{label}.png")
                fig.savefig(well_path, dpi=150, bbox_inches='tight')
            if show_plots:
                plt.show()
            else:
                plt.close(fig)

    return report_data

## 5. Run

Set `BATCH_MODE` / `WELLS_ONLY` / `input_path` / `BATCH_OUTPUT_DIR` in the Config cell (§1); set `DETECTION_METHOD` (`"edges"` or `"contour"`) below. Requires `roi_hints` from Section 3c. `BATCH_MODE=False` runs `input_path` alone with plots on; `True` sweeps the `SHIFT_SERIES` (same plate layout, since `roi_hints` is layout-specific), writing well/grid PNGs to `BATCH_OUTPUT_DIR/<plate_name>/`. `WELLS_ONLY=True` stops after well detection (skips Cellpose).

In [ ]:
# Which detector to use for the batch run — flip to "contour" if approach A wins the bake-off.
DETECTION_METHOD = "edges"


def process_plate(path, show_plots, save_dir=None, wells_only=False):
    plate_name = os.path.basename(path)
    print(f"Reading {plate_name}...")
    img = tifi.imread(path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB) if len(img.shape) == 2 else img[..., :3]

    # ROI-hinted detection: plate boxes re-detected per image inside the drawn ROIs
    # (roi_hints from Section 3c), wells placed by rows x cols grid + ring-refined.
    valid_wells, plate_boxes = detect_wells_from_rois(img_rgb, roi_hints, method=DETECTION_METHOD,
                                                      margin_frac=MARGIN_FRAC, well_r_frac=WELL_R_FRAC,
                                                      return_boxes=True)

    # Grid overlay (search area yellow, detected plate lime, wells red) — shown inline
    # and/or saved as grid.png per plate.
    if show_plots or save_dir:
        figure, axis = plt.subplots(figsize=(10, 10))
        render_overlay(axis, img_rgb, roi_hints, valid_wells, plate_name, plate_boxes=plate_boxes)
        if save_dir:
            grid_path = os.path.join(save_dir, plate_name, "grid.png")
            os.makedirs(os.path.dirname(grid_path), exist_ok=True)
            figure.savefig(grid_path, dpi=150, bbox_inches="tight")
        if show_plots:
            plt.show()
        else:
            plt.close(figure)

    if wells_only:
        return [{"Plate": plate_name, "Well": label, "x": x, "y": y, "r": r}
                for (x, y, r, label) in valid_wells]

    wells_dir = os.path.join(save_dir, plate_name) if save_dir else None
    return count_colonies(img_rgb, valid_wells, plate_name, show_plots=show_plots, save_dir=wells_dir)


if BATCH_MODE:
    # Wipe any previous batch_output before this run, so a well/grid PNG from a stale run
    # can never get mistaken for output from the current one.
    shutil.rmtree(BATCH_OUTPUT_DIR, ignore_errors=True)
    os.makedirs(BATCH_OUTPUT_DIR, exist_ok=True)

    # roi_hints is per plate-layout, so a batch runs one experiment series (same layout),
    # not a mixed folder. Sweep the validated SHIFT_SERIES.
    tif_paths = [os.path.join(ROI_INPUT_DIR, scan_name) for scan_name in SHIFT_SERIES]
    n_imgs = len(tif_paths)
    print(f"Batch mode: {n_imgs} plates ({DETECTION_METHOD} detector)")
    print(f"Writing well/grid images to {BATCH_OUTPUT_DIR}/<plate_name>/\n")
    report_data = []
    for img_idx, path in enumerate(tif_paths, start=1):
        print(f"\n=== [{img_idx}/{n_imgs}] {os.path.basename(path)} ===")
        # WELLS_ONLY is a diagnostic step — show inline even in batch
        report_data.extend(process_plate(
            path, show_plots=WELLS_ONLY, save_dir=BATCH_OUTPUT_DIR, wells_only=WELLS_ONLY
        ))
else:
    report_data = process_plate(input_path, show_plots=True, wells_only=WELLS_ONLY)

In [ ]:
# Convert results to a clean Pandas DataFrame
df = pd.DataFrame(report_data)

# Display the spreadsheet directly in the notebook for a final review
display(df)

# Save to disk: one combined CSV for batch runs, per-plate CSV for single-image runs;
# "wells" vs "Results" in the name so a wells-only run doesn't overwrite a full run's CSV.
# Batch runs write the CSV inside BATCH_OUTPUT_DIR so it travels with the well/grid PNGs
# in the zip below, instead of sitting separately in the repo root.
suffix = "wells" if WELLS_ONLY else "Results"
if BATCH_MODE:
    csv_name = os.path.join(BATCH_OUTPUT_DIR, f"batch_{suffix}.csv")
else:
    csv_name = f"{os.path.splitext(os.path.basename(input_path))[0]}_{suffix}.csv"
df.to_csv(csv_name, index=False)

print(f"💾 Data successfully saved to {csv_name}")

# Zip the whole batch_output folder (well/grid PNGs + CSV) with a timestamp, for sending
# to others — one self-contained file per run instead of a loose folder.
if BATCH_MODE:
    zip_stem = f"{BATCH_OUTPUT_DIR}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    zip_path = shutil.make_archive(zip_stem, "zip", BATCH_OUTPUT_DIR)
    print(f"📦 Batch output zipped to {zip_path}")